# Phase 3: CH-05 (Multi-Resolution Scaling) & CH-03 / A4 (Same-H Blending)

**Workshop Paper PARMA-DITAM 2027 · crossstage-stitch · Nhánh: `chuong/p3-execution-plan`**

Notebook này thực hiện hai chiến dịch thực nghiệm trọng tâm của Phase 3 trên GPU **NVIDIA Tesla T4** (`sm_75`):
1. **CH-05 — Synthetic Multi-Resolution Stress Scaling:** Đánh giá khả năng mở rộng ở 3 cấp độ phân giải chuẩn (**720p**, **1080p**, **4K**), đo đạc workspace bộ nhớ và độ trễ latency trên hai track độc lập: `s6_only` (khóa $H_{\text{gt}}$) và `end_to_end` (toàn bộ pipeline $S1\to S6$). Track này sử dụng dữ liệu synthetic và hoàn toàn độc lập với UDIS-D.
2. **CH-03 / Ablation A4 — Same-Homography Blending Trade-off:** Đánh giá sự đánh đổi giữa chất lượng và tài nguyên tính toán của 3 chế độ blending trong Stage 6 (**Feather**, **Multiband 3 mức**, **Multiband 5 mức**) khi cố định hoàn toàn cặp ảnh $RGB$, ma trận Homography $H$ gốc $B\to A$ và canvas. Tự động xuất bảng LaTeX và trích xuất ảnh đường nối (seam crops) định tính.

### Quy trình thực hiện (6 Phần tuần tự):
- **Phần 1 — Khởi tạo & GPU T4:** Kết nối Google Drive, giải nén mã nguồn mới nhất và kiểm tra GPU Tesla T4.
- **Phần 2 — Biên dịch Native Runners:** Build CMake Release cho `stress_multires_cuda` và `test_s6_cuda` trên `sm_75`.
- **Phần 3 — CH-05 Synthetic Stress-Test:** Sinh dữ liệu 720p/1080p/4K, chạy 2 campaign (`s6_only` & `end_to_end`), xuất báo cáo scaling và bảng LaTeX.
- **Phần 4 — CH-03/A4 Chuẩn bị Cohort & Khóa Registration:** Nạp cohort UDIS-D, khóa ma trận $H$ gốc và pre-registration trước khi render.
- **Phần 5 — CH-03/A4 Chạy 3 Blender & Xuất Kết Quả:** Chạy campaign native trên T4 với `--save-images`, xuất bảng telemetry LaTeX và trích xuất seam crops.
- **Phần 6 — Sao lưu Toàn bộ Bằng chứng:** Đóng gói toàn bộ artifacts thực nghiệm vào `MyDrive/p3_evidence/` để nghiệm thu.

## Phần 1. Khởi Tạo Workspace & Xác Nhận Phần Cứng GPU Tesla T4

Kết nối Google Drive, nạp mã nguồn từ `crossstage_stitch_source.zip` và kiểm tra GPU Tesla T4.

In [ ]:
# @title 1. Kết nối Drive, Nạp Mã Nguồn và Xác nhận GPU Tesla T4 { display-mode: "form" }
from google.colab import drive
import os, shutil, zipfile, subprocess, sys, json, time
from pathlib import Path

# 1. Kết nối Google Drive
drive.mount('/content/drive')
REPO_DIR = Path('/content/crossstage-stitch')
SOURCE_ZIP = Path('/content/drive/MyDrive/crossstage_stitch_source.zip')

# 2. Giải nén mã nguồn vào workspace
if SOURCE_ZIP.exists():
    print(f"Đang giải nén mã nguồn {SOURCE_ZIP.name} vào {REPO_DIR}...")
    with zipfile.ZipFile(SOURCE_ZIP, 'r') as zf:
        zf.extractall(REPO_DIR)
    print(f"✅ Đã cập nhật mã nguồn thành công ({len(list(REPO_DIR.rglob('*')))} tệp).")
else:
    print(f"⚠️ Không tìm thấy {SOURCE_ZIP}, sử dụng mã nguồn hiện tại trong {REPO_DIR}.")

os.chdir(REPO_DIR)

# 3. Khởi tạo Git repo cục bộ phục vụ provenance tracking nếu cần
if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "init", "-b", "chuong/p3-execution-plan"], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "config", "user.name", "chuong"], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "config", "user.email", "chuong@crossstage.local"], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "add", "."], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "commit", "-m", "chore: colab environment snapshot"], cwd=REPO_DIR, check=True)

# 4. Xác nhận phần cứng GPU bắt buộc là Tesla T4 (sm_75)
print("\n=== KIỂM TRA PHẦN CỨNG GPU TESLA T4 ===")
gpu_check = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"], capture_output=True, text=True, check=True)
gpu_info = gpu_check.stdout.strip()
print(f"GPU phát hiện: {gpu_info}")
assert any(keyword in gpu_info for keyword in ["Tesla T4", "T4"]), \
    f"LỖI MÔI TRƯỜNG: GPU hiện tại là '{gpu_info}', không phải Tesla T4! Môi trường chuẩn của workshop yêu cầu Tesla T4."
print("✅ Phần cứng GPU Tesla T4 hợp lệ.")

# 5. Chụp receipt môi trường
ENV_RECEIPT = Path("build/ch03_ch05_receipt.json")
ENV_RECEIPT.parent.mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable, "-m", "tools.env.generate_receipt", "--output", str(ENV_RECEIPT), "--verbose"], check=True)
print(f"✅ Đã tạo Environment Receipt: {ENV_RECEIPT}")


## Phần 2. Biên Dịch Native Runners Trên Kiến Trúc `sm_75` (Tesla T4)

Biên dịch bản Release của hai runner native:
- `stress_multires_cuda`: Runner phục vụ đo stress-test đa độ phân giải CH-05 (hỗ trợ capacity explicit và full/ROI verification).
- `test_s6_cuda`: Runner phục vụ đo độ trễ steady-state và workspace của S6 trong CH-03 / A4.

In [ ]:
# @title 2. Cấu hình CMake và Biên dịch Native Runners { display-mode: "form" }
import subprocess
from pathlib import Path

BUILD_DIR = Path("build")
BUILD_DIR.mkdir(parents=True, exist_ok=True)

print("=== BẮT ĐẦU CẤU HÌNH VÀ BIÊN DỊCH CMAKE RELEASE (sm_75) ===")
subprocess.run([
    "cmake", "-S", ".", "-B", str(BUILD_DIR),
    "-DCMAKE_BUILD_TYPE=Release",
    "-DCMAKE_CUDA_ARCHITECTURES=75",
    "-DCMAKE_CUDA_COMPILER=/usr/local/cuda/bin/nvcc"
], check=True)

print("\n--- Biên dịch mục tiêu stress_multires_cuda và test_s6_cuda ---")
subprocess.run([
    "cmake", "--build", str(BUILD_DIR),
    "--target", "stress_multires_cuda", "test_s6_cuda", "-j2"
], check=True)

# Tìm vị trí binary
candidates_mr = list(BUILD_DIR.glob("**/stress_multires_cuda"))
RUNNER_MULTIRES = candidates_mr[0] if candidates_mr else BUILD_DIR / "harness" / "stress_multires_cuda"

candidates_s6 = list(BUILD_DIR.glob("**/test_s6_cuda"))
RUNNER_S6 = candidates_s6[0] if candidates_s6 else BUILD_DIR / "src" / "s6" / "test_s6_cuda"

assert RUNNER_MULTIRES.exists(), f"LỖI: Không tìm thấy binary stress_multires_cuda tại {RUNNER_MULTIRES}"
assert RUNNER_S6.exists(), f"LỖI: Không tìm thấy binary test_s6_cuda tại {RUNNER_S6}"

print(f"\n✅ stress_multires_cuda sẵn sàng: {RUNNER_MULTIRES}")
print(f"✅ test_s6_cuda sẵn sàng:          {RUNNER_S6}")

# Chạy nhanh CTest kiểm tra correctness các module trọng yếu
print("\n--- Kiểm tra CTest nhanh (test_s6_cuda | test_admission_control) ---")
subprocess.run(["ctest", "--test-dir", str(BUILD_DIR), "-R", "test_s6_cuda|test_admission_control", "--output-on-failure"], check=True)
print("✅ Toàn bộ CTest kiểm tra an toàn bộ nhớ và kernel S6 đã PASS 100%.")


## Phần 3. CH-05: Synthetic Multi-Resolution Stress Scaling (720p, 1080p, 4K)

Quy trình thực nghiệm CH-05:
1. **Sinh dữ liệu synthetic:** 18 cặp ảnh chia đều cho 3 độ phân giải (1280×720, 1920×1080, 3840×2160), 3 mức overlap (30%, 50%, 70%) và 2 phép biến đổi phối cảnh (Affine, Projective).
2. **Campaign 1 (`s6_only`):** Đo lường riêng Stage 6 với ma trận $H_{\text{gt}}$ đã khóa, kiểm tra workspace memory allocation và runtime scaling trên 3 blender.
3. **Campaign 2 (`end_to_end`):** Đo toàn bộ pipeline $S1\to S6$ với cấu hình cố định, tính toán sai số góc chiếu (corner reprojection error) so với ground truth.
4. **Báo cáo:** Tự động tổng hợp bảng LaTeX `scaling_table.tex` và biểu đồ vector `workspace_scaling.svg`.

In [ ]:
# @title 3.1 Sinh Bộ Dữ Liệu Synthetic Multi-Resolution Chuẩn { display-mode: "form" }
import subprocess, sys, json
from pathlib import Path

CH05_DATA = Path("data/ch05")
if CH05_DATA.exists():
    shutil.rmtree(CH05_DATA)

print("=== BẮT ĐẦU SINH DỮ LIỆU SYNTHETIC MULTI-RES (720p, 1080p, 4K) ===")
subprocess.run([
    sys.executable, "-m", "tools.data_prep.generate_synthetic_multires",
    "--output", str(CH05_DATA),
    "--seed", "20261030",
    "--scenes", "1",
    "--overlaps", "0.3", "0.5", "0.7",
    "--warps", "affine", "projective",
    "--host-budget-mib", "256",
    "--chunk-rows", "64"
], check=True)

print("\n--- Xác minh tính bất biến dữ liệu (--verify-only) ---")
subprocess.run([
    sys.executable, "-m", "tools.data_prep.generate_synthetic_multires",
    "--output", str(CH05_DATA),
    "--verify-only"
], check=True)

manifest_data = json.loads((CH05_DATA / "data_manifest.json").read_text(encoding="utf-8"))
pairs_count = len(manifest_data.get("pairs", {}))
print(f"✅ Đã sinh và xác minh thành công {pairs_count} cặp ảnh synthetic chuẩn tại: {CH05_DATA}")


In [ ]:
# @title 3.2 Chạy Hai Campaign CH-05 trên Tesla T4 (s6_only & end_to_end) { display-mode: "form" }
import subprocess, sys, json
from pathlib import Path

CH05_MANIFEST = Path("data/ch05/data_manifest.json")
OUT_S6 = Path("results/ch05-s6-t4")
OUT_E2E = Path("results/ch05-e2e-t4")

if OUT_S6.exists(): shutil.rmtree(OUT_S6)
if OUT_E2E.exists(): shutil.rmtree(OUT_E2E)

# 1. Campaign s6_only
print("============================================================")
print("=== 1. CHẠY CAMPAIGN S6_ONLY (648 LƯỢT ĐO TRÊN TESLA T4) ===")
print("============================================================")
subprocess.run([
    sys.executable, "-m", "tools.bench.stress_multires",
    "--manifest", str(CH05_MANIFEST),
    "--track", "s6_only",
    "--runner", str(RUNNER_MULTIRES),
    "--output", str(OUT_S6),
    "--expected-sm", "75",
    "--warmups", "2",
    "--repeats", "10",
    "--seed", "20261030",
    "--host-budget-mib", "512",
    "--device-reserve-mib", "256",
    "--timeout", "300"
], check=True)

print("\n--- Xác minh bất biến Campaign s6_only (--verify-only) ---")
subprocess.run([sys.executable, "-m", "tools.bench.stress_multires", "--output", str(OUT_S6), "--verify-only"], check=True)
print(f"✅ Campaign s6_only hoàn tất và được xác minh tại: {OUT_S6}")

# 2. Campaign end_to_end
print("\n============================================================")
print("=== 2. CHẠY CAMPAIGN END_TO_END (648 LƯỢT ĐO TRÊN TESLA T4) ===")
print("============================================================")
subprocess.run([
    sys.executable, "-m", "tools.bench.stress_multires",
    "--manifest", str(CH05_MANIFEST),
    "--track", "end_to_end",
    "--runner", str(RUNNER_MULTIRES),
    "--output", str(OUT_E2E),
    "--expected-sm", "75",
    "--scale", "0.5",
    "--keypoints", "1000",
    "--ransac", "256",
    "--warmups", "2",
    "--repeats", "10",
    "--seed", "20261030",
    "--host-budget-mib", "512",
    "--device-reserve-mib", "256",
    "--timeout", "300"
], check=True)

print("\n--- Xác minh bất biến Campaign end_to_end (--verify-only) ---")
subprocess.run([sys.executable, "-m", "tools.bench.stress_multires", "--output", str(OUT_E2E), "--verify-only"], check=True)
print(f"✅ Campaign end_to_end hoàn tất và được xác minh tại: {OUT_E2E}")


In [ ]:
# @title 3.3 Xuất Báo Cáo Nghiệm Thu & Bảng LaTeX Scaling CH-05 { display-mode: "form" }
import subprocess, sys, json
from pathlib import Path

REPORT_S6 = Path("results/ch05-s6-t4-report")
REPORT_E2E = Path("results/ch05-e2e-t4-report")

if REPORT_S6.exists(): shutil.rmtree(REPORT_S6)
if REPORT_E2E.exists(): shutil.rmtree(REPORT_E2E)

print("=== XUẤT BÁO CÁO NGHIỆM THU CH-05 CHO CẢ HAI TRACK ===")
subprocess.run([
    sys.executable, "-m", "tools.analysis.generate_multires_report",
    "--campaign", "results/ch05-s6-t4", "--output", str(REPORT_S6)
], check=True)
subprocess.run([sys.executable, "-m", "tools.analysis.generate_multires_report", "--campaign", str(REPORT_S6), "--verify-only"], check=True)

subprocess.run([
    sys.executable, "-m", "tools.analysis.generate_multires_report",
    "--campaign", "results/ch05-e2e-t4", "--output", str(REPORT_E2E)
], check=True)
subprocess.run([sys.executable, "-m", "tools.analysis.generate_multires_report", "--campaign", str(REPORT_E2E), "--verify-only"], check=True)

print("\n============================================================")
print("=== BẢNG KẾT QUẢ LATENCY & WORKSPACE SCALING (s6_only) ===")
print("============================================================")
tex_file = REPORT_S6 / "scaling_table.tex"
if tex_file.exists():
    print(tex_file.read_text(encoding="utf-8"))

print("\n🎉 CH-05 ĐÃ HOÀN TẤT ĐẦY ĐỦ VÀ SẴN SÀNG NGHIỆM THU!")


## Phần 4. CH-03 / Ablation A4: Same-Homography Blending Trade-off

Quy trình thực nghiệm A4:
1. **Chuẩn bị cohort UDIS-D:** Chọn lọc 10 cặp ảnh UDIS-D có độ phức tạp cao, độ phủ overlap lớn ($>85\%$) và đã được xác thực ma trận Homography $H$ chuẩn từ Reference của CH-01.
2. **Khóa Pre-Registration:** Cố định tuyệt đối $RGB$, 9 hệ số ma trận $H$ gốc $B\to A$, canvas bounds và footprint masks trước khi render.
3. **Chạy Native trên Tesla T4:** Thực thi 3 chế độ (**Feather**, **Multiband 3**, **Multiband 5**) với 10 repeats + 2 warmups mỗi mode, kích hoạt cờ `--save-images` để lưu panorama phục vụ kiểm tra đường nối.
4. **Xuất bảng LaTeX & Seam Crops:** Tự động tổng hợp bảng so sánh độ trễ / workspace và trích xuất ảnh crop lossless độ phân giải gốc tại vị trí đường biên chồng lấp.

In [ ]:
# @title 4.1 Khôi phục Data (nếu cần), Chuẩn bị Cohort và Khóa Manifest A4 { display-mode: "form" }
import json, shutil, zipfile
from pathlib import Path

# 1. Kiểm tra ảnh training PPM trong data/p3/training/images/
P3_DIR = Path("data/p3")
P3_BUNDLE = Path("/content/drive/MyDrive/p3_protocol_bundle.zip")
IMG_SAMPLE = P3_DIR / "training" / "images" / "000792_a.ppm"

if not IMG_SAMPLE.exists():
    if P3_BUNDLE.exists():
        print(f"Đang khôi phục data/p3/ từ {P3_BUNDLE}...")
        with zipfile.ZipFile(P3_BUNDLE, 'r') as zf:
            zf.extractall(P3_DIR)
        print("✅ Đã khôi phục data/p3/ thành công.")
    else:
        raise FileNotFoundError(f"Không tìm thấy ảnh tại {IMG_SAMPLE} và không có bundle {P3_BUNDLE} trên Drive!")

# 2. Cohort 10 cặp ảnh UDIS-D đại diện đã thẩm định homography gốc B->A (độ phủ overlap 85% - 99%)
VERIFIED_A4_COHORT = {
    "000792": {"overlap": 259739, "H": [[0.1619340088, 0.0006140107, -0.1104412126], [0.0022090297, 0.1622506532, -1.0], [-3.407e-07, 2.1265e-06, 0.1614410606]]},
    "002674": {"overlap": 243510, "H": [[0.0457401619, -0.000945388, -1.0], [0.000723042, 0.045800833, -0.0245273859], [1.7383e-06, 6.541e-07, 0.0451576908]]},
    "003309": {"overlap": 242282, "H": [[0.0312597384, 2.8259e-06, -1.0], [0.0001571107, 0.0308115205, 0.1577641564], [3.755e-07, -5.149e-07, 0.0311895999]]},
    "010157": {"overlap": 241030, "H": [[0.0248157705, 0.0014641863, -0.2976681747], [-0.0001016615, 0.0261282344, -1.0], [-1.6497e-06, 4.1388e-06, 0.0242181916]]},
    "009477": {"overlap": 240034, "H": [[0.0240716575, -0.0001559325, 1.0], [-0.0001315087, 0.0229927445, -0.1102576163], [-5.164e-07, 4.978e-07, 0.0223699359]]},
    "009423": {"overlap": 239321, "H": [[0.085660326, 0.0160033722, -1.0], [-0.0005946832, 0.0868240603, 0.6317751508], [7.08e-08, -1.65582e-05, 0.0891522505]]},
    "009598": {"overlap": 238154, "H": [[0.0189714439, -0.0014406256, 1.0], [0.0005263028, 0.0206289707, -0.1577696579], [-4.5647e-06, -3.735e-07, 0.0225711038]]},
    "009583": {"overlap": 229036, "H": [[0.0229553125, 0.000384352, -1.0], [0.0005427501, 0.0228599895, -0.3288075248], [2.5052e-06, 3.788e-07, 0.0222217098]]},
    "009015": {"overlap": 227797, "H": [[0.0154203899, 0.0002504391, 1.0], [-0.0013561203, 0.0167476303, 0.3229721246], [-4.566e-06, -6.94e-08, 0.0178481028]]},
    "010285": {"overlap": 225702, "H": [[0.0202297754, -0.0027266689, 0.5894646139], [-0.0012989617, 0.0202692488, 1.0], [-3.1339e-06, 7.73e-07, 0.022412242]]}
}

A4_DATA_DIR = Path("data/a4")
A4_DATA_DIR.mkdir(parents=True, exist_ok=True)

manifest_a4 = {
    "schema_version": "s6-same-h-v1",
    "evidence_type": "real_images",
    "split": "development",
    "pairs": {}
}

for pid, data in VERIFIED_A4_COHORT.items():
    img_a = f"../p3/training/images/{pid}_a.ppm"
    img_b = f"../p3/training/images/{pid}_b.ppm"
    assert (A4_DATA_DIR / img_a).resolve().exists(), f"Thiếu ảnh {img_a}"
    assert (A4_DATA_DIR / img_b).resolve().exists(), f"Thiếu ảnh {img_b}"
    manifest_a4["pairs"][f"pair_{pid}"] = {
        "a": img_a,
        "b": img_b,
        "H": data["H"]
    }

manifest_path = A4_DATA_DIR / "manifest.json"
manifest_path.write_text(json.dumps(manifest_a4, indent=2), encoding="utf-8")
print(f"✅ Đã xuất manifest s6-same-h-v1 thành công ({len(manifest_a4['pairs'])} cặp) tại: {manifest_path}")


In [ ]:
# @title 4.2 Khóa Pre-Registration Trước Khi Render Native { display-mode: "form" }
import subprocess, sys
from pathlib import Path

OUT_LOCK = Path("results/a4-lock-t4")
if OUT_LOCK.exists(): shutil.rmtree(OUT_LOCK)

print("=== BẮT ĐẦU KHÓA REGISTRATION CHO ABLATION A4 ===")
subprocess.run([
    sys.executable, "-m", "tools.bench.same_h",
    "--manifest", "data/a4/manifest.json",
    "--output", str(OUT_LOCK),
    "--prepare-only",
    "--warmups", "2",
    "--repeats", "10",
    "--seed", "20261030"
], check=True)

print("\n--- Xác minh tính bất biến của Registration (--verify-only) ---")
subprocess.run([
    sys.executable, "-m", "tools.bench.same_h",
    "--output", str(OUT_LOCK),
    "--verify-only"
], check=True)

print(f"✅ Registration đã khóa thành công tại: {OUT_LOCK / 'registration.json'}")


## Phần 5. Chạy Campaign A4 Trên Tesla T4 & Xuất Bảng / Seam Crops

Thực thi native runner `test_s6_cuda` trên GPU Tesla T4 cho 10 cặp ảnh qua 3 chế độ (Feather, Multiband 3, Multiband 5). Kích hoạt `--save-images` để trích xuất ảnh đường nối trực quan.

In [ ]:
# @title 5.1 Thực thi Campaign A4 trên Tesla T4 (--save-images) { display-mode: "form" }
import subprocess, sys
from pathlib import Path

OUT_A4 = Path("results/a4-t4")
if OUT_A4.exists(): shutil.rmtree(OUT_A4)

print("============================================================")
print("=== BẮT ĐẦU CHẠY CAMPAIGN A4 TRÊN TESLA T4 (360 LƯỢT ĐO) ===")
print("============================================================")
subprocess.run([
    sys.executable, "-m", "tools.bench.same_h",
    "--manifest", "data/a4/manifest.json",
    "--registration", "results/a4-lock-t4/registration.json",
    "--output", str(OUT_A4),
    "--runner", str(RUNNER_S6),
    "--warmups", "2",
    "--repeats", "10",
    "--seed", "20261030",
    "--save-images"
], check=True)

print("\n--- Xác minh bất biến Campaign A4 (--verify-only) ---")
subprocess.run([
    sys.executable, "-m", "tools.bench.same_h",
    "--output", str(OUT_A4),
    "--verify-only"
], check=True)

print(f"✅ Toàn bộ 3 blender trên 10 cặp ảnh đã render hoàn tất tại: {OUT_A4}")


In [ ]:
# @title 5.2 Xuất Bảng Telemetry LaTeX và Trích xuất Seam Crops Định tính { display-mode: "form" }
import subprocess, sys
from pathlib import Path

OUT_TABLES = Path("results/a4-tables")
OUT_VISUALS = Path("results/a4-visuals")

if OUT_TABLES.exists(): shutil.rmtree(OUT_TABLES)
if OUT_VISUALS.exists(): shutil.rmtree(OUT_VISUALS)

# 1. Sinh bảng tổng hợp telemetry
print("=== 1. TỔNG HỢP BẢNG LATENCY & WORKSPACE A4 ===")
subprocess.run([
    sys.executable, "-m", "tools.analysis.generate_a4_table",
    "--campaign", "results/a4-t4",
    "--output", str(OUT_TABLES)
], check=True)
subprocess.run([sys.executable, "-m", "tools.analysis.generate_a4_table", "--campaign", str(OUT_TABLES), "--verify-only"], check=True)

tex_a4 = OUT_TABLES / "a4_table.tex"
if tex_a4.exists():
    print("\n--- BẢNG LATEX A4 (FEATHER VS MULTIBAND) ---")
    print(tex_a4.read_text(encoding="utf-8"))

# 2. Trích xuất ảnh crop đường nối seam crops
print("\n=== 2. TRÍCH XUẤT SEAM CROPS ĐỊNH TÍNH ===")
subprocess.run([
    sys.executable, "-m", "tools.analysis.extract_seam_crops",
    "--campaign", "results/a4-t4",
    "--output", str(OUT_VISUALS)
], check=True)
subprocess.run([sys.executable, "-m", "tools.analysis.extract_seam_crops", "--campaign", str(OUT_VISUALS), "--verify-only"], check=True)

png_crops = list(OUT_VISUALS.glob("*.png"))
print(f"\n✅ Đã trích xuất thành công {len(png_crops)} tệp ảnh seam crops tại: {OUT_VISUALS}")
for p in sorted(png_crops)[:6]:
    print(f"  - {p.name}")
if (OUT_VISUALS / "captions.txt").exists():
    print("\n--- Captions khách quan ---")
    print((OUT_VISUALS / "captions.txt").read_text(encoding="utf-8"))

print("\n🎉 CH-03 / A4 ĐÃ HOÀN TẤT ĐẦY ĐỦ VÀ SẴN SÀNG NGHIỆM THU!")


## Phần 6. Đóng Gói và Sao Lưu Toàn Bộ Bằng Chứng Lên Google Drive

Lưu trữ toàn bộ artifacts vào thư mục độc lập `MyDrive/p3_evidence/<timestamp>_ch03_ch05/`.

In [ ]:
# @title 6. Đóng Gói & Sao Lưu Gói Bằng Chứng CH-03 & CH-05 lên Google Drive { display-mode: "form" }
import shutil, datetime
from pathlib import Path

timestamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d_%H%M%SZ")
DRIVE_EVIDENCE_ROOT = Path("/content/drive/MyDrive/p3_evidence")
DEST = DRIVE_EVIDENCE_ROOT / f"run_{timestamp}_ch03_ch05"
DEST.mkdir(parents=True, exist_ok=True)

print(f"Đang sao lưu kết quả thực nghiệm vào: {DEST}...")

# Copy các thư mục kết quả chính
items_to_copy = [
    ("build/ch03_ch05_receipt.json", DEST / "environment_receipt.json"),
    ("results/ch05-s6-t4-report", DEST / "ch05-s6-report"),
    ("results/ch05-e2e-t4-report", DEST / "ch05-e2e-report"),
    ("results/ch05-s6-t4/multires_scaling_metrics.csv", DEST / "ch05_s6_metrics.csv"),
    ("results/ch05-e2e-t4/multires_scaling_metrics.csv", DEST / "ch05_e2e_metrics.csv"),
    ("results/a4-tables", DEST / "a4-tables"),
    ("results/a4-visuals", DEST / "a4-visuals"),
    ("results/a4-t4/a4_same_h_metrics.csv", DEST / "a4_same_h_metrics.csv"),
    ("results/a4-t4/report.json", DEST / "a4_report.json"),
    ("results/a4-t4/registration.json", DEST / "a4_registration.json")
]

for src_str, dst_path in items_to_copy:
    src = Path(src_str)
    if src.is_dir():
        if dst_path.exists(): shutil.rmtree(dst_path)
        shutil.copytree(src, dst_path)
        print(f"  -> Đã lưu thư mục: {dst_path.name}/")
    elif src.is_file():
        shutil.copy2(src, dst_path)
        print(f"  -> Đã lưu tệp:     {dst_path.name}")

# Nén thành file bundle ZIP tổng hợp
zip_out = DRIVE_EVIDENCE_ROOT / f"p3_ch03_ch05_bundle_{timestamp}"
shutil.make_archive(str(zip_out), 'zip', root_dir=str(DEST))
print(f"\n✅ ĐÃ TẠO GÓI ARCHIVE THỰC NGHIỆM TẠI: {zip_out}.zip")
print(f"\n🎉 HOÀN TẤT THỰC NGHIỆM CH-03 VÀ CH-05 TRÊN GOOGLE COLAB TESLA T4!")
